<a href="https://colab.research.google.com/github/nabillahumi/S5-Mesin_Learning/blob/main/Jobsheet%2013%20PBL/Tugas%20Prakrikum%20(JS13)%20PBL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Langkah 1: Load Dataset Kafe & Review Serta Pra-pemrosesan Data

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.utils import to_categorical

# Memuat dataset cafes.csv dan reviews.csv
cafes_df = pd.read_csv('cafes.csv').dropna(subset=['rating', 'review_count', 'latitude', 'longitude'])
reviews_df = pd.read_csv('reviews.csv').dropna(subset=['rating', 'text'])

# Menggabungkan fitur kafe dengan agregasi review untuk membangun dataset klasifikasi multi-kelas
reviews_agg = reviews_df.groupby('cafe_id').agg(
    avg_review_rating=('rating', 'mean'),
    total_reviews=('review_id', 'count')
).reset_index()

merged_df = pd.merge(cafes_df, reviews_agg, on='cafe_id', how='inner')

# Menyiapkan Fitur (X) dan Target Kategori Multi-Kelas (y) (misal: 3 kelas kategori rating)
X_raw = merged_df[['rating', 'review_count', 'avg_review_rating', 'total_reviews', 'latitude', 'longitude']].values
ratings = merged_df['rating'].values

# Membuat label kelas (0, 1, 2)
y = np.zeros(len(ratings), dtype=int)
y[ratings >= 4.6] = 1
y[ratings >= 4.8] = 2

# Normalisasi fitur
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

# Split data training dan testing (80:20)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# One-hot encoding label target (3 kelas)
y_train_encoded = to_categorical(y_train, num_classes=3)
y_test_encoded = to_categorical(y_test, num_classes=3)

print("Shape X_train:", X_train.shape)
print("Shape y_train_encoded:", y_train_encoded.shape)

Shape X_train: (75, 6)
Shape y_train_encoded: (75, 3)


#Langkah 2: Membangun Model JST dengan 2 Hidden Layer (Konfigurasi Dasar)

In [3]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

print("--- LANGKAH 2: MEMBANGUN MODEL DENGAN 2 HIDDEN LAYER ---")

model_dasar = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)), # Hidden Layer 1
    Dense(32, activation='relu'),                                 # Hidden Layer 2
    Dense(3, activation='softmax')                                # Output layer (3 kelas)
])

# Kompilasi model
model_dasar.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
print("Model dasar berhasil dibangun!")

--- LANGKAH 2: MEMBANGUN MODEL DENGAN 2 HIDDEN LAYER ---
Model dasar berhasil dibangun!


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


#Langkah 3 & 4: Melatih Model (Training) dan Evaluasi Akurasi

In [4]:
import time

print("--- LANGKAH 3 & 4: LATIH MODEL DAN EVALUASI ---")

start_time = time.time()
history_dasar = model_dasar.fit(X_train, y_train_encoded, epochs=50, batch_size=8, verbose=0)
train_time_dasar = time.time() - start_time

# Evaluasi model
loss_dasar, acc_dasar = model_dasar.evaluate(X_test, y_test_encoded, verbose=0)

print(f"Waktu Pelatihan (Model Dasar) : {train_time_dasar:.4f} detik")
print(f"Akurasi pada Data Uji        : {acc_dasar:.4f} ({acc_dasar * 100:.2f}%)")

--- LANGKAH 3 & 4: LATIH MODEL DAN EVALUASI ---
Waktu Pelatihan (Model Dasar) : 4.7129 detik
Akurasi pada Data Uji        : 0.8947 (89.47%)


#Eksperimen Parameter Lain
1. Ubah Jumlah Neuron di Hidden Layer (Misal: 256 dan 128)
Python

In [5]:
print("--- EKSPERIMEN 1: UBAH JUMLAH NEURON (256 & 128) ---")

model_neuron = Sequential([
    Dense(256, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(128, activation='relu'),
    Dense(3, activation='softmax')
])

model_neuron.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

start_time = time.time()
history_neuron = model_neuron.fit(X_train, y_train_encoded, epochs=50, batch_size=8, verbose=0)
train_time_neuron = time.time() - start_time

loss_neuron, acc_neuron = model_neuron.evaluate(X_test, y_test_encoded, verbose=0)
print(f"Akurasi (Neuron 256/128): {acc_neuron:.4f} | Waktu: {train_time_neuron:.4f} detik")

--- EKSPERIMEN 1: UBAH JUMLAH NEURON (256 & 128) ---
Akurasi (Neuron 256/128): 0.8947 | Waktu: 4.3915 detik


2. Tambahkan Satu Hidden Layer Lagi (Total 3 Hidden Layer)

In [6]:
print("--- EKSPERIMEN 2: TAMBAH SATU HIDDEN LAYER LAGI ---")

model_3layer = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(3, activation='softmax')
])

model_3layer.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

start_time = time.time()
history_3layer = model_3layer.fit(X_train, y_train_encoded, epochs=50, batch_size=8, verbose=0)
train_time_3layer = time.time() - start_time

loss_3layer, acc_3layer = model_3layer.evaluate(X_test, y_test_encoded, verbose=0)
print(f"Akurasi (3 Hidden Layers): {acc_3layer:.4f} | Waktu: {train_time_3layer:.4f} detik")

--- EKSPERIMEN 2: TAMBAH SATU HIDDEN LAYER LAGI ---
Akurasi (3 Hidden Layers): 0.8947 | Waktu: 4.4133 detik


3. Eksperimen Fungsi Aktivasi: Sigmoid vs ReLU

In [7]:
print("--- EKSPERIMEN 3: AKTIVASI SIGMOID VS RELU ---")

# Model dengan Sigmoid
model_sigmoid = Sequential([
    Dense(64, activation='sigmoid', input_shape=(X_train.shape[1],)),
    Dense(32, activation='sigmoid'),
    Dense(3, activation='softmax')
])

model_sigmoid.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

start_time = time.time()
history_sig = model_sigmoid.fit(X_train, y_train_encoded, epochs=50, batch_size=8, verbose=0)
train_time_sig = time.time() - start_time

loss_sig, acc_sig = model_sigmoid.evaluate(X_test, y_test_encoded, verbose=0)
print(f"Akurasi (Sigmoid): {acc_sig:.4f} | Waktu: {train_time_sig:.4f} detik")
print(f"Akurasi (ReLU)   : {acc_dasar:.4f} | Waktu: {train_time_dasar:.4f} detik")

--- EKSPERIMEN 3: AKTIVASI SIGMOID VS RELU ---
Akurasi (Sigmoid): 0.7895 | Waktu: 6.1075 detik
Akurasi (ReLU)   : 0.8947 | Waktu: 4.7129 detik


4. Tabulasi Perbandingan Akurasi dan Waktu Pelatihan

In [8]:
print("--- TABULASI PERBANDINGAN EKSPERIMEN JST (DATASET CAFIND) ---")

comparison_results = [
    {'Konfigurasi Model': 'Dasar (64, 32 / ReLU)', 'Akurasi': acc_dasar, 'Waktu (detik)': train_time_dasar},
    {'Konfigurasi Model': 'Neuron Besar (256, 128 / ReLU)', 'Akurasi': acc_neuron, 'Waktu (detik)': train_time_neuron},
    {'Konfigurasi Model': '3 Hidden Layers (128, 64, 32 / ReLU)', 'Akurasi': acc_3layer, 'Waktu (detik)': train_time_3layer},
    {'Konfigurasi Model': 'Sigmoid (64, 32 / Sigmoid)', 'Akurasi': acc_sig, 'Waktu (detik)': train_time_sig},
]

df_comparison = pd.DataFrame(comparison_results)
display(df_comparison)

--- TABULASI PERBANDINGAN EKSPERIMEN JST (DATASET CAFIND) ---


,Konfigurasi Model,Akurasi,Waktu (detik)
0,"Dasar (64, 32 / ReLU)",0.894737,4.712860
1,"Neuron Besar (256, 128 / ReLU)",0.894737,4.391479
2,"3 Hidden Layers (128, 64, 32 / ReLU)",0.894737,4.413294
3,"Sigmoid (64, 32 / Sigmoid)",0.789474,6.107506
